## Step 4  
Inputs:  
- s3://thesis--ec331-s3/melted-price-bids/
- s3://thesis--ec331-s3/melted-volume-bids/  

Output: s3://thesis--ec331-s3/merged-price-volume-bids/  

Problems:
- I don't think that there is Settlement date in volume bids, I believe it is now trading bids

In [4]:
import boto3

# Initialize S3 client
s3_client = boto3.client('s3')

# Parse the S3 URI
bucket_name = "thesis--ec331-s3"
file_key = "melted-volume-bids/duids_1066.txt"

# Read the file
response = s3_client.get_object(Bucket=bucket_name, Key=file_key)
file_content = response['Body'].read().decode('utf-8')

print(file_content)

AAAAAAAAADAAAAFN0
AAAAAAAAAFAAAAVW5
AAAAAAAAANAAAATE9
AAAAAAAAAgAAABDb21
AAAAAAAAXAAAATWF4
AAAAAAAAoAAABCQU5
AAAAAAAAsAAABCQU5
AAAAAAACwAAAEFnZ3
AAAAAAAHAAAAFRlY2
AAAAAADgAAAEJJRE9
AAAAAAYAAAAUmVnX0
AAAAAAz6
AAAAAEAAAAAAAAAA0
AAAAAEAAAAAAAAAA4
AAAAAGAAAAUmVnaW9
AAAABAAAAAz3
AAAABUAAABGdWVsX1
AAAAHgAAAA4
AAAANAAAARGlzcGF0
AAABFTkFCTEVNRU5
AAABGdWVsX1
AAABNYXhfQ2FwX2
AAABNYXhfUk9DL01
AAABSZWdfQ2FwX2
AAABUZWNobm9sb2d5
AAACAAAAENhdGVnb3
AAACAAz9
AAACABz8
AAACACz7
AAACADz3
AAACAEz8
AAACAFz7
AAACAHj3
AAACAHz8
AAACAIz7
AAACAKj9
AAACAKz8
AAACALz7
AAACANz8
AAACAOz7
AAAENsYXNzaWZpY2
AAAFBhcnRpY2
AAAIA0P7
AAAIAEP7
AAAIAQP7
AAAIAcP7
AAAIAdP3
AAAIAoP7
AAAQAAAAAAAKAA4
AAAgC49
AACQAAAEZJWEVETE9
AADQAAAEVOQUJMRU1
AAEAAAAAYAAABwYW5
AAEAIAAOQBAAC0
AAFAJAAAgCQAA9
AAJAcAAOwGAAC0
AAMAAAAUkFNUERPV05
AMAAEQDAAAQAwAA1
AQAAHQEAAA4
AUAAKwFAAB4
AYAAEgGAAAUBgAA4
BDYXAgY29uc3
BQkxFTUVOVE1
BRAAAAM7
BRCIsICJwYW5
BWEFWQUlMAAAAAP7
CBTb3VyY2
DRVIOT02
DY0IiwgIm1
EQVZBSUw0
EQVZBSUw1
EQVZBSUw2AAB6
EQVZBSUw3
EQVZBSUw4
EQVZ

In [ ]:
import dask.dataframe as dd
import pandas as pd

def main():
    # 1. Define your S3 paths (wildcard to read multiple files).
    volume_path = "s3://thesis--ec331-s3/melted-volume-bids/*.parquet"
    price_path = "s3://thesis--ec331-s3/melted-price-bids/*.parquet"
    output_path = "s3://thesis--ec331-s3/merged-price-volume-bids/"
    
    # 2. Read both datasets as Dask DataFrames
    volume_ddf = dd.read_parquet(volume_path, engine="pyarrow")
    price_ddf = dd.read_parquet(price_path, engine="pyarrow")
    
    print("Volume columns:", volume_ddf.columns)
    print("Volume partitions:", volume_ddf.npartitions)
    print("Price columns:", price_ddf.columns)
    print("Price partitions:", price_ddf.npartitions)
    
    # 3. Convert 'SETTLEMENTDATE' to datetime (optional)
    volume_ddf["SETTLEMENTDATE"] = dd.to_datetime(volume_ddf["SETTLEMENTDATE"], errors="coerce")
    price_ddf["SETTLEMENTDATE"] = dd.to_datetime(price_ddf["SETTLEMENTDATE"], errors="coerce")
    
    # 4. Merge on common keys
    merge_keys = ["SETTLEMENTDATE", "DUID", "BIDTYPE", "BIDBAND"]
    merged_ddf = volume_ddf.merge(
        price_ddf,
        on=merge_keys,
        how="inner",
        suffixes=("_volume", "_price"),
        # shuffle="disk"  # Uncomment if low on memory
    )
    
    # Optional: Repartition to reduce the number of output files
    # merged_ddf = merged_ddf.repartition(npartitions=30)
    
    # 5. Write out merged result to S3 in Parquet format
    merged_ddf.to_parquet(output_path, engine="pyarrow", write_index=False)
    
    print(f"Done! Merged data is written to {output_path}")

if __name__ == "__main__":
    main()